# Lab 13 — Stress-Testing Models
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Week 13 lab · about 75 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/labs/lab13-stress-testing-models/Lab13_Stress_Testing_Models.ipynb)

A model that scores well on its test set can still fail in the field: a photo taken in shade, a building shape it never saw, a group of bridges it rarely learned from. Today you **stress-test** three models from earlier labs, the way a careful engineer load-tests a new design, and then write a **model card** — a one-page spec sheet that says what a model is for and where it breaks.

### What you will be able to do
1. Measure how two crack classifiers (pretrained features vs. raw pixels) hold up under brightness, blur, shadow, stains, occlusion, and compression.
2. Show what happens when a model is asked to extrapolate beyond its training range, and add a range check.
3. Run a subgroup error analysis on the bridge model (by owner, material, age, and traffic).
4. (Optional) Check whether predicted probabilities can be trusted, and compute permutation feature importance.
5. Write a one-page model card using the NIST AI RMF functions: Govern, Map, Measure, Manage.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `concrete_cracks_sample.zip` (CC BY 4.0), `building_energy_efficiency.csv` (CC BY 4.0), `arizona_bridges_nbi2024.csv` (U.S. Government work).

**AI-use label:** **AI-Assisted** — you may ask an AI tool for explanations or debugging help. Fill in the AI-use acknowledgment at the end. You are responsible for every line.

### The NIST AI Risk Management Framework in one table
The U.S. National Institute of Standards and Technology (NIST) published the **AI Risk Management Framework
(AI RMF 1.0)** in 2023. It is voluntary guidance organized into four functions. Today is mostly **Measure**.

| Function | Plain meaning | Today |
|---|---|---|
| **Govern** | Policies, roles, and accountability for AI in the organization | Who owns the model card |
| **Map** | Understand the context: users, uses, and what could go wrong | Intended use, out-of-scope uses |
| **Measure** | Test and quantify risks: accuracy, robustness, subgroup errors | Parts 1–4 |
| **Manage** | Act on what you measured: limits, safeguards, monitoring | Range checks, thresholds, human review |

### Plan for today (project check-ins happen during this lab)
| Part | What | Status |
|---|---|---|
| 1 | Two crack classifiers under stress | **Core** (25 min) |
| 2 | Energy model asked to extrapolate | **Core** (10 min) |
| 3 | Bridge model: errors by owner, material, age, traffic | **Core** (15 min) |
| 3b | Can we trust the predicted probabilities? (calibration) | Optional |
| 4 | Permutation feature importance | Optional |
| 5 | Model card | **Core** (10 min) |
| ⭐ | Challenge: train on State bridges only | Optional |

If your team is called for its project check-in, skip the optional parts and come back to them later.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## Part 1 — Two crack classifiers under stress (25 min)
We build **two** crack classifiers on the same 200 training photos and test both on the same 120 test photos:
- **Model A — pretrained features (Lab 06):** **ResNet-18**, pretrained on 1.2 million ImageNet photos, turns each
  photo into 512 numbers (features); a **logistic regression** learns crack vs. no crack from them.
- **Model B — raw pixels (the Week 13 lecture demo):** shrink each photo to 48 × 48 gray pixels (2,304 numbers,
  0 = black, 1 = white) and fit a logistic regression directly on those numbers.

The question is not "which is better on clean photos" but **"which one keeps working when the photo changes?"**

In [ ]:
import zipfile, glob, random, io
import torch, torchvision
from torchvision import transforms
from PIL import Image, ImageEnhance, ImageFilter, ImageDraw, ImageStat
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

torch.manual_seed(42)
zipfile.ZipFile(get_data("concrete_cracks_sample.zip")).extractall(".")
crack = sorted(glob.glob("concrete_cracks/crack/*.jpg"))
intact = sorted(glob.glob("concrete_cracks/no_crack/*.jpg"))
rng = random.Random(42)
rng.shuffle(crack); rng.shuffle(intact)

N_TRAIN, N_TEST = 100, 60       # photos per class
train_files = crack[:N_TRAIN] + intact[:N_TRAIN]
test_files = crack[N_TRAIN:N_TRAIN + N_TEST] + intact[N_TRAIN:N_TRAIN + N_TEST]
y_train_img = np.array([1] * N_TRAIN + [0] * N_TRAIN)     # 1 = crack, 0 = no crack
y_test_img = np.array([1] * N_TEST + [0] * N_TEST)
train_imgs = [Image.open(f).convert("RGB") for f in train_files]
test_imgs = [Image.open(f).convert("RGB") for f in test_files]
print(len(train_imgs), "training photos,", len(test_imgs), "test photos")

The ResNet-18 weights are about 45 MB (model downloads happen on Google's computers, not on your laptop or phone data plan).

In [ ]:
weights = torchvision.models.ResNet18_Weights.DEFAULT
resnet = torchvision.models.resnet18(weights=weights)
resnet.fc = torch.nn.Identity()          # remove the 1000-class ImageNet head -> 512 features per image
resnet.eval()
prep = transforms.Compose([transforms.Resize(224), transforms.ToTensor(),
                           transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def features(images):
    """Model A inputs: a (n_images x 512) array of ResNet-18 features."""
    out = []
    with torch.no_grad():
        for i in range(0, len(images), 32):
            batch = torch.stack([prep(im) for im in images[i:i + 32]])
            out.append(resnet(batch).numpy())
    return np.vstack(out)

def pixels(images):
    """Model B inputs: a (n_images x 2304) array of 48 x 48 gray pixel values between 0 and 1."""
    return np.array([np.asarray(im.convert("L").resize((48, 48), Image.BILINEAR), dtype=np.float32).ravel() / 255
                     for im in images])

model_a = LogisticRegression(max_iter=2000).fit(features(train_imgs), y_train_img)
model_b = LogisticRegression(C=0.1, max_iter=5000).fit(pixels(train_imgs), y_train_img)
print(f"Clean test photos — Model A (ResNet-18 features): {accuracy_score(y_test_img, model_a.predict(features(test_imgs))):.1%}")
print(f"Clean test photos — Model B (raw pixels):         {accuracy_score(y_test_img, model_b.predict(pixels(test_imgs))):.1%}")

Those are the numbers a vendor would put on a brochure. Now we change the photos the way the field does.
**PIL** (the Python imaging library) has simple tools for this:
- `brightness(img, f)` — f < 1 darker (dusk, under a deck), f > 1 brighter (Arizona noon sun, wet glare)
- `blur(img, r)` — out of focus or motion blur from a drone; r = blur radius in pixels
- `shadow(img, s)` — a soft-edged shadow (say, of a girder) over part of the surface, darkened by a fraction s
- `stain(img, s)` — a dark blotch in the middle (water stain, oil, efflorescence shadow), darkened by s
- `occlude(img, f)` — something covers the center (a tag, debris, a plant); f = side length as a share of the photo
- `jpeg(img, q)` — heavy JPEG compression, as from a phone app or a cheap camera; q = quality (100 = original)

In [ ]:
def brightness(img, factor):
    return ImageEnhance.Brightness(img).enhance(factor)

def blur(img, radius):
    return img.filter(ImageFilter.GaussianBlur(radius)) if radius > 0 else img

def shadow(img, strength):
    """Darken a diagonal region (like the shadow of a girder) by `strength` (0 = none, 0.95 = very dark)."""
    if strength == 0:
        return img
    W, H = img.size
    mask = Image.new("L", (W, H), 0)
    ImageDraw.Draw(mask).polygon([(0, 0), (int(0.65 * W), 0), (int(0.35 * W), H), (0, H)], fill=255)
    mask = mask.filter(ImageFilter.GaussianBlur(6))                 # soft shadow edge
    return Image.composite(brightness(img, 1 - strength), img, mask)

def stain(img, strength):
    """A soft dark oval in the middle of the photo, darkened by `strength`."""
    if strength == 0:
        return img
    W, H = img.size
    mask = Image.new("L", (W, H), 0)
    ImageDraw.Draw(mask).ellipse([int(0.15 * W), int(0.25 * H), int(0.85 * W), int(0.75 * H)], fill=255)
    return Image.composite(brightness(img, 1 - strength), img, mask.filter(ImageFilter.GaussianBlur(10)))

def occlude(img, frac):
    """Cover a square in the center (side = frac x photo width) with the photo's average color."""
    if frac == 0:
        return img
    W, H = img.size
    s = int(frac * W)
    out = img.copy()
    fill = tuple(int(v) for v in ImageStat.Stat(img).mean)
    ImageDraw.Draw(out).rectangle([(W - s) // 2, (H - s) // 2, (W + s) // 2, (H + s) // 2], fill=fill)
    return out

def jpeg(img, quality):
    """Save and reload the photo as a JPEG at the given quality (100 = keep the original)."""
    if quality >= 100:
        return img
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

sample = test_imgs[0]
examples = [("original", sample), ("brightness 0.2", brightness(sample, 0.2)), ("brightness 3", brightness(sample, 3)),
            ("blur r = 12", blur(sample, 12)), ("stain 0.5", stain(sample, 0.5)), ("occlude 0.5", occlude(sample, 0.5)),
            ("JPEG q = 5", jpeg(sample, 5)), ("shadow 0.7 (no crack)", shadow(test_imgs[-1], 0.7))]
fig, axes = plt.subplots(1, 8, figsize=(17, 2.7))
for ax, (title, im) in zip(axes, examples):
    ax.imshow(im); ax.set_title(title, fontsize=9); ax.axis("off")
plt.suptitle("One cracked photo under field conditions (last: an uncracked surface with a shadow)")
plt.tight_layout(); plt.show()

The helper below applies one change at several strengths to **all 120 test photos** and records, for
both models, the accuracy, the share of cracks still found (**crack recall**), and the share of clean surfaces
wrongly flagged (**false alarms**). Each call takes about a minute on Colab's CPU (Model A has to recompute ResNet
features). **Scale check:** with 120 test photos, one photo = 0.8 percentage points of accuracy — differences of
1–2 points are noise.

In [ ]:
def stress_test(perturb, levels):
    """For each level: accuracy, crack recall, and false-alarm rate of both models on the changed test photos."""
    rows = []
    for level in levels:
        changed = [perturb(im, level) for im in test_imgs]
        for name, pred in [("A: ResNet features", model_a.predict(features(changed))),
                           ("B: raw pixels", model_b.predict(pixels(changed)))]:
            rows.append({"level": level, "model": name,
                         "accuracy": accuracy_score(y_test_img, pred),
                         "crack_recall": pred[y_test_img == 1].mean(),
                         "false_alarms": pred[y_test_img == 0].mean()})
    return pd.DataFrame(rows)

results = {}
results["Brightness factor (1 = original)"] = stress_test(brightness, [0.1, 0.2, 0.35, 0.5, 1.0, 2.0, 3.0, 4.0])
results["Brightness factor (1 = original)"].round(2)

> 💡 **Hint:** Copy the brightness line: `stress_test(blur, [0, 2, 4, 8, 12, 16])` — pass the function name without parentheses.

In [ ]:
# TODO: stress-test blur (radii 0, 2, 4, 8, 12, 16) and shadow (strengths 0, 0.5, 0.7, 0.85, 0.95).
results["Blur radius (pixels)"] = None
results["Shadow strength (fraction darkened)"] = None
display(results["Blur radius (pixels)"].round(2))

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert all(results.get(k) is not None for k in ['Blur radius (pixels)', 'Shadow strength (fraction darkened)'])
print('Correct!')

Three harsher kinds of field damage (run and observe — about 3 minutes). To see any full table later, run `results["Stain strength (fraction darkened)"].round(2)` (or another key) in a new cell.

In [ ]:
results["Stain strength (fraction darkened)"] = stress_test(stain, [0, 0.3, 0.5, 0.7])
results["Occluded share of width"] = stress_test(occlude, [0, 0.25, 0.5, 0.75])
results["JPEG quality (100 = original)"] = stress_test(jpeg, [100, 30, 10, 5, 2])
display(results["Occluded share of width"].round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5), sharey=True)
for ax, (xlabel, res) in zip(axes.ravel(), results.items()):
    for name, color, marker in [("A: ResNet features", "#0C234B", "o"), ("B: raw pixels", "#AB0520", "s")]:
        r = res[res["model"] == name]
        ax.plot(r["level"], r["accuracy"], marker=marker, color=color, label=name)
    ax.axhline(0.5, color="gray", ls=":", lw=1.5)
    ax.set_xlabel(xlabel); ax.set_ylim(0.3, 1.02); ax.grid(alpha=0.3)
    if "JPEG" in xlabel:
        ax.invert_xaxis()                     # left = original, right = most compressed
    ax.text(ax.get_xlim()[0], 0.52, " coin flip", color="gray", fontsize=9)
for ax in axes[:, 0]:
    ax.set_ylabel("Accuracy on 120 test photos (0–1)")
axes[0, 0].legend(loc="lower right", fontsize=9)
fig.suptitle("Same photos, same changes, two models: pretrained features vs. raw pixels")
plt.tight_layout(); plt.show()

**✍️ Question:** Compare the two models. Under which changes does Model B (raw pixels) collapse to a coin flip, and what does it predict when it fails (look at crack recall and false alarms in the tables)? Where does Model A (ResNet features) finally break, and how? Remember: 1 photo = 0.8 points.

*Your answer:* 

**✍️ Question:** Name two field conditions on a real bridge inspection that look like these changes, and one thing you would add to the model's instructions for use (a **Manage** action).

*Your answer:* 

In [ ]:
del resnet
import gc; gc.collect()      # free memory before the next part

## Part 2 — Asking a model to extrapolate (10 min)
In Lab 05 you predicted **heating load** (kWh/m² per year) from building shape. Here is a realistic mistake:
train the model only on the **more compact** buildings (relative compactness ≥ 0.75), then use it on the less
compact ones. We compare two models: linear regression and a random forest.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

energy = pd.read_csv(get_data("building_energy_efficiency.csv"))
inputs = ["relative_compactness", "surface_area_m2", "wall_area_m2", "roof_area_m2", "overall_height_m",
          "orientation", "glazing_area_ratio", "glazing_distribution"]
target = "heating_load_kwh_m2"

inside = energy[energy["relative_compactness"] >= 0.75]      # the "training world"
outside = energy[energy["relative_compactness"] < 0.75]      # buildings the model never saw
train_e, test_in = train_test_split(inside, test_size=0.25, random_state=42)
print(len(train_e), "training buildings;", len(test_in), "in-range test;", len(outside), "out-of-range")

lin = LinearRegression().fit(train_e[inputs], train_e[target])
rf_e = RandomForestRegressor(n_estimators=200, random_state=42).fit(train_e[inputs], train_e[target])
for name, m in [("Linear regression", lin), ("Random forest", rf_e)]:
    mae_in = mean_absolute_error(test_in[target], m.predict(test_in[inputs]))
    print(f"{name:18s} in-range test MAE = {mae_in:.2f} kWh/m²")

> 💡 **Hint:** Same as the loop above, but with `outside[target]` and `m.predict(outside[inputs])` for each model.

In [ ]:
# TODO: compute the MAE of both models on the `outside` buildings.
mae_out_lin = None
mae_out_rf = None
print(f"Out-of-range MAE: linear {mae_out_lin:.1f} kWh/m², random forest {mae_out_rf:.1f} kWh/m²")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert mae_out_lin is not None and mae_out_lin > 20 and mae_out_rf > 10
print('Error jumps — as expected.')

Always compare with a **baseline** — the simplest reasonable prediction. Here: ignore the inputs and
predict the *average training building's* heating load for every new building.

In [ ]:
mae_out_mean = mean_absolute_error(outside[target], np.full(len(outside), train_e[target].mean()))
print(f"Baseline 'predict the training mean ({train_e[target].mean():.1f} kWh/m²)': out-of-range MAE {mae_out_mean:.1f} kWh/m²")
print(f"Linear regression: {mae_out_lin:.1f} kWh/m²   Random forest: {mae_out_rf:.1f} kWh/m²")

In [ ]:
plt.figure(figsize=(6, 5))
for m, name, color in [(lin, "linear", "#AB0520"), (rf_e, "random forest", "#378DBD")]:
    plt.scatter(outside[target], m.predict(outside[inputs]), s=12, color=color, alpha=0.6, label=f"{name}, out of range")
plt.scatter(test_in[target], rf_e.predict(test_in[inputs]), s=12, color="#0C234B", label="random forest, in range")
lims = [0, 120]
plt.plot(lims, lims, "k--", lw=1, label="perfect prediction")
plt.xlabel("Actual heating load (kWh/m² per year)"); plt.ylabel("Predicted heating load (kWh/m² per year)")
plt.title("Inside its training range the model is excellent; outside, it is not")
plt.legend(fontsize=8); plt.show()

print(energy.groupby(energy["relative_compactness"] >= 0.75)["overall_height_m"].unique())

**✍️ Question:** Why do the two models fail differently, and how do they compare with the mean baseline? (Hint: look at the last printed line — what is the overall height of every training building?)

*Your answer:* 

**A simple Manage action: a range check.** Before trusting a prediction, check that every input lies
inside the range the model was trained on.

> 💡 **Hint:** Loop over `inputs` and keep a name `c` when `row[c] < train_min[c] or row[c] > train_max[c]`. A list comprehension works: `[c for c in inputs if ...]`.

In [ ]:
# TODO: return a list of input names whose value in `row` is below the training minimum or above the maximum.
train_min, train_max = train_e[inputs].min(), train_e[inputs].max()

def out_of_range(row):
    """List the inputs of one building (a pandas row) that fall outside the training range."""
    return None

print("In-range building:    ", out_of_range(test_in.iloc[0]))
print("Out-of-range building:", out_of_range(outside.iloc[0]))

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert out_of_range(test_in.iloc[0]) == [] and 'overall_height_m' in out_of_range(outside.iloc[0])
print('Correct!')

## Part 3 — Subgroup error analysis on the bridge model (15 min)
An overall score can hide groups where a model is much worse. We rebuild the Lab 11 bridge model (same features,
same random forest) and ask: **is it equally good for every kind of bridge — and for every bridge owner?**

Bridges are owned by different agencies: the state DOT, counties and cities, the Bureau of Indian Affairs and
tribal governments, federal agencies such as the U.S. Forest Service. Owners differ in budgets, staff, and
inspection resources. The model does **not** use owner as an input, but its errors can still differ by owner —
which matters, because the bridges it misses belong to real communities.

**One change from Lab 11:** Lab 11's target, *poor* (deck rating ≤ 4), has only 41 bridges — split across owners
or materials, that leaves 0–16 per group, far too few to measure anything. For this audit we use a **low deck**
target, **rating ≤ 5** (fair-to-poor, 399 bridges), the same definition as the Week 13 lecture demo.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import recall_score

df = pd.read_csv(get_data("arizona_bridges_nbi2024.csv"))
bridges = df[df["deck_condition"].notna()].copy()                 # same steps as Lab 11...
bridges["low"] = (bridges["deck_condition"] <= 5).astype(int)     # ...except the target: rating <= 5
bridges["age_years"] = 2024 - bridges["year_built"]
recon = bridges["year_reconstructed"].where(bridges["year_reconstructed"] <= 2024)
bridges["years_since_work"] = 2024 - recon.fillna(bridges["year_built"])
bridges["pct_truck_traffic"] = bridges["pct_truck_traffic"].fillna(bridges["pct_truck_traffic"].median())
numeric = ["age_years", "years_since_work", "avg_daily_traffic", "pct_truck_traffic", "lanes_on",
           "main_spans", "max_span_m", "length_m", "deck_width_m"]
categorical = ["material", "design_type"]
X = pd.get_dummies(bridges[numeric + categorical], columns=categorical, dtype=int)
y = bridges["low"]

def make_forest():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=42)
print(X.shape, "-", y.sum(), "low decks (rating <= 5)")

# Before any model: how common are poor (<= 4) and low (<= 5) decks for each owner group?
bridges.groupby("owner_group").agg(bridges=("low", "size"), low_decks=("low", "sum"), share_low=("low", "mean"),
                                   share_poor=("deck_condition", lambda s: (s <= 4).mean())).round(3)

To give every bridge a fair prediction we use **k-fold cross-validation** with k = 5: split the bridges
into 5 parts, train on 4, predict the 5th, and rotate, so every bridge gets a prediction from a model that never
saw it. All 399 low decks are tested (a single test set would hold only 120). We flag a bridge when the predicted
probability is ≥ 0.5 (the default threshold).

*Why your numbers differ from the Week 13 slides:* the lecture demo used one 70/30 train/test split and slightly
different inputs and forest settings; cross-validation tests every bridge, so its counts are larger and steadier.
The pattern across groups is the same.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
proba = cross_val_predict(make_forest(), X, y, cv=cv, method="predict_proba")[:, 1]
bridges["flag"] = (proba >= 0.5).astype(int)
print(f"Overall: recall {recall_score(y, bridges['flag']):.0%} "
      f"({bridges.loc[bridges['low'] == 1, 'flag'].sum()} of {y.sum()} low decks), "
      f"{bridges['flag'].sum()} of {len(bridges)} bridges flagged")

# Groups to audit
bridges["material_group"] = bridges["material"].str.replace(" (continuous)", "", regex=False)
bridges["material_group"] = bridges["material_group"].replace({"Wood or timber": "Wood/masonry", "Masonry": "Wood/masonry"})
bridges["age_group"] = pd.cut(bridges["age_years"], [0, 25, 50, 75, 200], labels=["0-25 yr", "26-50 yr", "51-75 yr", "76+ yr"])
bridges["traffic_group"] = pd.cut(bridges["avg_daily_traffic"], [-1, 999, 9999, 1e9],
                                  labels=["< 1,000 veh/day", "1,000-9,999", "10,000+"])

In [ ]:
def group_report(column):
    """Recall and false-alarm rate of the flags, for each group in `column`."""
    rows = []
    for g, d in bridges.groupby(column, observed=True):
        low = d["low"] == 1
        rows.append({column: g, "bridges": len(d), "low decks": int(low.sum()),
                     "caught": int(d.loc[low, "flag"].sum()),
                     "recall": d.loc[low, "flag"].mean() if low.any() else np.nan,
                     "false-alarm rate": d.loc[~low, "flag"].mean(),
                     "share flagged": d["flag"].mean()})
    return pd.DataFrame(rows).set_index(column)

by_owner = group_report("owner_group")
by_owner.round(2)

*Recall* is blank (NaN) for a group with no low decks — you cannot miss what isn't there. Groups with
only a handful of bridges ("Other", "Private or railroad") are too small to judge.

> 💡 **Hint:** Call `group_report("material_group")`, and the same for the other two columns.

In [ ]:
# TODO: make the same report for material_group, age_group, and traffic_group.
by_material = None
by_age = None
by_traffic = None
display(by_material.round(2)); display(by_age.round(2)); display(by_traffic.round(2))

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert by_material is not None and len(by_material) == 4 and len(by_age) == 4 and len(by_traffic) == 3
print('Correct!')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5), sharey=True)
panels = [(by_owner, "Owner group"), (by_material, "Main material"), (by_age, "Bridge age"),
          (by_traffic, "Average daily traffic")]
for ax, (rep, title) in zip(axes.ravel(), panels):
    rep = rep[rep["low decks"] > 0]                      # only groups that have low decks
    xs = np.arange(len(rep))
    ax.bar(xs - 0.2, rep["recall"], 0.4, color="#AB0520", label="recall (low decks caught)")
    ax.bar(xs + 0.2, rep["false-alarm rate"], 0.4, color="#B0B7BC", label="false-alarm rate")
    ax.set_xticks(xs)
    ax.set_xticklabels([f"{i}\n({p} low / {n})" for i, p, n in zip(rep.index, rep["low decks"], rep["bridges"])],
                       fontsize=8)
    ax.set_title(title); ax.grid(axis="y", alpha=0.3)
for ax in axes[:, 0]:
    ax.set_ylabel("Fraction (0–1)")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Low-deck model errors by subgroup (5-fold cross-validation, threshold 0.5; groups with ≥ 1 low deck)")
plt.tight_layout(); plt.show()

**✍️ Question:** Compare the owner groups. Which owner's bridges are flagged most often, and is that mostly correct flags or false alarms? Which owner's low decks are missed most often? Who bears the cost of each kind of error?

*Your answer:* 

**✍️ Question:** Before running this, most people guess the model will be worst for the owner with the most low decks. Which **material** and which **age** group actually have the worst recall? Why might the model miss those low decks, and why are those misses worrying?

*Your answer:* 

**✍️ Question:** Should `owner_group` be added as an input to the model? Give one argument for and one against.

*Your answer:* 

### Part 3b (optional) — Can we trust the predicted probabilities?
A model is **calibrated** if, among bridges it gives a probability of about 0.7, about 70% really have a low deck.
We group the cross-validated probabilities into bins (0–0.1, 0.1–0.2, …) and compare the average predicted
probability with the share of bridges that are actually low. Points on the diagonal = calibrated.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

lr_balanced = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))  # as in Lab 11
lr_plain = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))                             # no class weights
proba_lr_bal = cross_val_predict(lr_balanced, X, y, cv=cv, method="predict_proba")[:, 1]
proba_lr_plain = cross_val_predict(lr_plain, X, y, cv=cv, method="predict_proba")[:, 1]

def reliability(p):
    bins = pd.cut(p, np.linspace(0, 1, 11), include_lowest=True)
    t = pd.DataFrame({"p": p, "y": y.values, "bin": bins}).groupby("bin", observed=True)
    t = t.agg(predicted=("p", "mean"), actually_low=("y", "mean"), bridges=("y", "size"))
    return t[t["bridges"] >= 10]            # skip nearly empty bins

plt.figure(figsize=(5.5, 5))
for p, name, color in [(proba, "random forest (balanced)", "#0C234B"),
                       (proba_lr_bal, "logistic regression (balanced, Lab 11)", "#AB0520"),
                       (proba_lr_plain, "logistic regression (no class weights)", "#378DBD")]:
    r = reliability(p)
    plt.plot(r["predicted"], r["actually_low"], "o-", color=color, label=name)
plt.plot([0, 1], [0, 1], "k--", lw=1, label="perfectly calibrated")
plt.xlabel("Average predicted probability of a low deck")
plt.ylabel("Share of bridges actually low (0–1)")
plt.title("Reliability diagram, 5-fold cross-validation")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()
reliability(proba).round(2)

**✍️ Question:** (Optional) When the balanced random forest says about 0.75, how often is the deck really low? Why are both class-weighted models overconfident, and what does that mean for using these numbers as "risk of a low deck"?

*Your answer:* 

## Part 4 (optional) — Permutation feature importance (10 min)
Lab 11 used the random forest's built-in importance. A more direct test: **shuffle one column** of the test data
(break its link to the target) and measure how much the model's score drops. A big drop means the model really
relies on that input. `sklearn.inspection.permutation_importance` does this, repeated 10 times per column.

For the score we use **average precision** (`"average_precision"`): a 0–1 summary of the precision–recall curve
from Lab 11 — it rewards ranking low decks above the others and is not fooled by the large "not low" majority.
We keep the low-deck (≤ 5) target, so the test set holds 120 low decks instead of Lab 11's 12 poor ones.

In [ ]:
from sklearn.inspection import permutation_importance

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)   # same split recipe as Lab 11
forest = make_forest().fit(X_tr, y_tr)
builtin = pd.Series(forest.feature_importances_, index=X.columns)

> 💡 **Hint:** `permutation_importance(forest, X_te, y_te, scoring="average_precision", n_repeats=10, random_state=42)`.

In [ ]:
# TODO: run permutation_importance on the TEST set with scoring="average_precision", n_repeats=10, random_state=42.
perm = None
perm_imp = pd.Series(perm.importances_mean, index=X.columns)
perm_std = pd.Series(perm.importances_std, index=X.columns)

In [ ]:
top = perm_imp.sort_values(ascending=False).head(10).index[::-1]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
axes[0].barh(top, builtin[top], color="#1E5288")
axes[0].set_xlabel("Built-in importance (share of total)"); axes[0].set_title("Random forest built-in")
axes[1].barh(top, perm_imp[top], xerr=perm_std[top], color="#AB0520")
axes[1].axvline(0, color="k", lw=0.8)
axes[1].set_xlabel("Drop in average precision when shuffled"); axes[1].set_title("Permutation (test set, ±1 std)")
plt.suptitle("Two views of what the bridge model relies on (top 10 by permutation)")
plt.tight_layout(); plt.show()
print(pd.DataFrame({"built-in": builtin, "permutation": perm_imp}).sort_values("permutation", ascending=False).head(8).round(3))

**✍️ Question:** Compare the two charts. Which features are on top in both? In Lab 11 (poor ≤ 4, only 29 poor decks to train on) `deck_width_m` was the top built-in feature — where is it now? What does that tell you about trusting an importance ranking from very few positive examples?

*Your answer:* 

## Part 5 — Write a model card (10 min)
A **model card** (Mitchell et al., 2019) is a short document that travels with a model, like a product data
sheet. Fill in the template below for the **bridge low-deck model** you audited today (mention Lab 11's poor-deck
version where useful). Keep it to one page: short bullets with numbers beat long paragraphs.

**✍️ Double-click the next cell and replace every "…" with your text.**

# Model card: <model name>
**Version / date:** … · **Owner (who is accountable):** …

## 1. Intended use
- Users and decision supported: …
- **Out of scope** (do not use for): …

## 2. Data
- Training data (source, years, size, how the target was defined): …
- Known gaps (what is missing or under-represented): …

## 3. Performance (Measure)
- Test metrics with numbers, compared with a baseline: …
- Subgroup results (where it is better / worse): …
- Stress tests (shift, extrapolation, perturbations): …

## 4. Risks and limitations (Map)
- What could go wrong, and who would be affected: …

## 5. Controls (Manage) and oversight (Govern)
- Rules for use (thresholds, human review, input range checks): …
- Monitoring and update plan; who signs off: …

---
## ⭐ Optional challenge — a real distribution shift (not graded)
Suppose a consultant built the model using only **State** bridges (the best-documented ones) and a county wants
to use it on its own bridges. Train on State bridges, test on everyone else, and compare with the
cross-validated recall from Part 3, where every owner group was in the training data.

In [ ]:
state = (bridges["owner_group"] == "State").values
shift_model = make_forest().fit(X[state], y[state])
others = bridges[~state].copy()
others["shift_flag"] = shift_model.predict(X[~state])
for g, d in others.groupby("owner_group"):
    low = d["low"] == 1
    if low.any():
        print(f"{g:15s} {low.sum():3d} low decks | trained on State only: recall {d.loc[low, 'shift_flag'].mean():.0%} "
              f"| Part 3 cross-validation: recall {d.loc[low, 'flag'].mean():.0%}")
low_o = others["low"] == 1
print(f"All non-State owners: {others.loc[low_o, 'shift_flag'].mean():.0%} vs {others.loc[low_o, 'flag'].mean():.0%}")

**✍️ Question:** (Challenge) How much recall is lost when the model never saw non-State bridges? What would you require before a county uses a model trained on someone else's bridges?

*Your answer:* 

---
## 🎟️ Exit ticket
Three short questions you answer **alone, without AI (AI-Free)**, during the last 10 minutes of lab — even if you have not finished every part above. Use your own words, 1–3 sentences each. (If you must leave early, answer them on your own before you submit.)

**1.** Give one example each of a robustness problem, an extrapolation problem, and a subgroup problem from today.

*Your answer:* 

**2.** Why can a model have a good overall score and still be unsafe to use for some bridges?

*Your answer:* 

**3.** Name the four NIST AI RMF functions and say which one a range check belongs to.

*Your answer:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
**Due:** Sunday 11:59 pm (Arizona time) of this week, on Brightspace. Late work: see the syllabus late-work table (two grace days per semester, then −10% per day, up to 3 days).

1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.